# Analysis of the final database

This notebook tries to identify characteristics or similarities between the proteins that do have reactions from both TCDB and Rhea.

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots
import plotly.io as pio
import itertools
import ast
import re
from collections import Counter

In [2]:
common = pd.read_csv("../1+2/tcdb_rhea_common_reactions.tsv", sep="\t")
tcids_common = common["TCID"]

In [3]:
parsed_data = []
for entry in tcids_common:
    parts = entry.split(".")
    if len(parts) >= 3:
        new_entry = ".".join(parts[:3])
        parsed_data.append(new_entry)

df = pd.DataFrame(parsed_data, columns=["Class_Subclass_Family"])
df[["Class", "Subclass", "Family"]] = df["Class_Subclass_Family"].str.split(".", expand=True)


fig = px.sunburst(
    df,
    path=["Class", "Subclass", "Family"],
    title="Protein Composition by Class, Subclass, and Family",
    width=700,
    height=700)

fig

Interesting finding on 1.B, very few transporters there have reactions for both Rhea and TCDB.

Also, an analysis of the complete database used in the pipeline, TDB, is of an even more obvious importance...

In [4]:
tdb = pd.read_csv("../1+2/transporters_df.tsv", sep="\t")
no_reactions = tdb[tdb["Reaction"] == "[('no_reaction_identified', 'no_reaction_identified')]"]
reactions = tdb[tdb["Reaction"] != "[('no_reaction_identified', 'no_reaction_identified')]"]

In [5]:
def tcid_sunburst(dfs, titles):

    all_classes = set()
    for df in dfs:
        for tc_id in df["TCID"]:
            class_ = tc_id.split(".")[0]
            all_classes.add(class_)

    all_classes = sorted(all_classes)

    palette = itertools.cycle(px.colors.qualitative.Set3)
    color_map = {cls: next(palette) for cls in all_classes}

    fig = make_subplots(rows=1, cols=len(dfs),
        specs=[[{"type": "domain"}] * len(dfs)],
        subplot_titles=titles)

    for i, df in enumerate(dfs):
        tc_data = []

        for tc_id in df["TCID"]:
            parts = tc_id.split(".")
            class_ = parts[0]
            subclass = ".".join(parts[:2])
            family = ".".join(parts[:3])
            tc_data.append({"Class": class_, "Subclass": subclass, "Family": family})
        
        df_out = pd.DataFrame(tc_data)
        sunburst = px.sunburst(df_out, path=["Class", "Subclass", "Family"], color="Class", color_discrete_map=color_map)
        fig.add_trace(sunburst.data[0], row=1, col=i + 1)
    
    fig.update_layout(
        height=600,
        width=1000)

    for i in range(len(dfs)):
        fig.layout.annotations[i].font.size = 20
    pio.write_image(fig, "reactions_tdb.pdf", format='pdf')
    fig.show()

tcid_sunburst([reactions, no_reactions], ["<b>a)</b>\tReactions<br>(9327 transporters)", "<b>b)</b>\tNo reactions<br>(14791 transporters)"])

In [ ]:
reactions["Reaction"] = reactions["Reaction"].apply(ast.literal_eval)
reactions["Reaction count"] = reactions["Reaction"].apply(len)

filtered = reactions[reactions["Reaction count"] <= 10]

reactions["Reaction count crop"] = reactions["Reaction count"].apply(lambda x: str(x) if x < 8 else "8+")

count_dist = reactions["Reaction count crop"].value_counts()
count_dist = count_dist.reindex(sorted(count_dist.index, key=lambda x: int(x.replace("+", ""))))

dist_df = count_dist.reset_index()
dist_df.columns = ["Reactions", "Transporters"]

fig = px.bar(
    dist_df,
    x="Reactions",
    y="Transporters",
    labels={"Number of Reactions": "Reactions per Transporter", "Transporters": "Transporter Count"},
    text="Transporters",
    color_discrete_sequence=["lightblue"])

fig.update_traces(textposition="outside", textfont=dict(size=20))
fig.update_layout(yaxis_title="Transporter Count", xaxis_title="Reactions per Transporter")

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        showgrid=False
    ),
    yaxis=dict(
        title_font=dict(size=20),
        tickfont=dict(size=16),
        gridcolor="black",
        zerolinecolor="black",
        range=[0,4800]
    ),
    margin=dict(l=60, r=40, t=60, b=60))

fig.show()
pio.write_image(fig, "reaction_pr_transporter.pdf", format="pdf")

A stacked bar chart might be handy to display the amount of transporters with reactions in each class!

In [7]:
tdb = pd.read_csv("../1+2/transporters_df.tsv", sep="\t")
tdb["ReactionPresent"] = tdb["Reaction"] != "[('no_reaction_identified', 'no_reaction_identified')]"
tdb["Class"] = tdb["TCID"].apply(lambda x: x.split(".")[0])
class_counts = tdb.groupby(["Class", "ReactionPresent"]).size().reset_index(name="Count")
class_counts["Reaction?"] = class_counts["ReactionPresent"].map({
    True: "YES",
    False: "NO"})

fig = px.bar(class_counts, x="Class", y="Count", color="Reaction?",
    color_discrete_map={
        "YES": "#90ee90",
        "NO": "#ff9999"})

fig.update_layout(
    barmode="stack",
    plot_bgcolor="white",
    xaxis_title="TC Class",
    yaxis_title="Number of Transporters",
    font=dict(size=16),
    title_font_size=20,
    yaxis=dict(gridcolor="black", linecolor="black", zerolinecolor="black"),
    xaxis=dict(linecolor="black")
)

fig.show()
pio.write_image(fig, "tc_class_reaction.pdf", format="pdf")

c:\Users\landr\AppData\Local\Programs\Python\Python310\lib\site-packages\plotly\express\_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



Want to see how many different reactions are given in TDB!

In [17]:
tdb = pd.read_csv("../1+2/transporters_df.tsv", sep="\t")

def extract_chebi_sets(reaction_str):
    chebi_sets = []
    reaction_list = ast.literal_eval(reaction_str)
    for _, chebi_str in reaction_list:
        chebis = set(re.findall(r"CHEBI:\d+", chebi_str))
        chebi_sets.append(chebis)
    return chebi_sets

tdb["Substrate Sets"] = tdb["Reaction"].apply(extract_chebi_sets)

all_sets = []
for list_of_sets in tdb["Substrate Sets"]:
    all_sets.extend(list_of_sets)

unique_substrate_sets = list({frozenset(s) for s in all_sets})

length_counts = Counter(len(s) for s in unique_substrate_sets)
for length, count in sorted(length_counts.items()):
    print(f"{count} substrate sets with {length} substrates")

fours = []
for el in unique_substrate_sets:
    if len(el) == 4:
        fours.append(el)

target_chebis = {"CHEBI:15422", "CHEBI:16761", "CHEBI:18367"}
count = sum(1 for s in unique_substrate_sets if target_chebis.issubset(s))

print(f"Found {count} sets containing all target ChEBIs (ATP, ADP, Phosphate group).")


1 substrate sets with 0 substrates
841 substrate sets with 1 substrates
1229 substrate sets with 2 substrates
95 substrate sets with 3 substrates
769 substrate sets with 4 substrates
183 substrate sets with 5 substrates
249 substrate sets with 6 substrates
14 substrate sets with 7 substrates
2 substrate sets with 8 substrates
Found 593 sets containing all target ChEBIs (ATP, ADP, Phosphate group).


In [ ]:
for s in unique_substrate_sets:
    if len(s) == 8:
        print(s)

frozenset({'CHEBI:58045', 'CHEBI:57287', 'CHEBI:57286', 'CHEBI:30616', 'CHEBI:15378', 'CHEBI:136842', 'CHEBI:33019', 'CHEBI:456215'})
frozenset({'CHEBI:58349', 'CHEBI:57783', 'CHEBI:16480', 'CHEBI:15377', 'CHEBI:32682', 'CHEBI:15378', 'CHEBI:57743', 'CHEBI:15379'})


<!-- Want to find the reactions with many substrates to see what they actually are... -->

Misc. less important evaluations below

In [10]:
either_df = pd.read_csv("../1+2/tcdb_rhea_either_reactions.tsv", sep="\t")
filtered_either_df = either_df[
    (either_df["Rhea:Reaction:CHEBI"].notna()) &
    (either_df["TCDB:Reaction:CHEBI"] == "[nan]")
]

In [21]:
all_df = pd.read_csv("../1+2/all.tsv", sep="\t")
merged_df = pd.merge(
    either_df,
    all_df[["AID", "TCID", "AA"]],
    on="AID",
    how="outer",
    suffixes=("_filtered", "_eq")
)

merged_df.drop_duplicates(subset=["AID", "TCID_eq", "AA_eq"], keep="first", inplace=True)
merged_df["TCID_filtered"] = merged_df["TCID_filtered"].fillna(merged_df["TCID_eq"])
merged_df["AA_filtered"] = merged_df["AA_filtered"].fillna(merged_df["AA_eq"])
merged_df.drop(columns=["TCID_eq", "AA_eq"], inplace=True)
merged_df.rename(columns={"TCID_filtered": "TCID","AA_filtered": "AA"}, inplace=True)

In [20]:
for col in ["TCDB:Reaction:CHEBI", "Rhea:Reaction:CHEBI"]:
    merged_df[col] = merged_df[col].apply(lambda x: np.nan if x == "[nan]" else x)


merged_df["TCDB"] = merged_df["TCDB:Reaction:CHEBI"].apply(
    lambda x: isinstance(x, str) and len(x) > 0
)

merged_df["RHEA"] = merged_df["Rhea:Reaction:CHEBI"].apply(
    lambda x: isinstance(x, str) and len(x) > 0
)

# Notably, all reactions from TCDB are already identified for all subunits. Not Rhea. An
# increase in 1308 new subunits or AAs.
tcdb_mapping = merged_df.groupby("TCID")["TCDB"].max()
merged_df["TCDB"] = merged_df["TCID"].map(tcdb_mapping)

rhea_mapping = merged_df.groupby("TCID")["RHEA"].max()
merged_df["RHEA"] = merged_df["TCID"].map(rhea_mapping)